<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
</head>
<body>
    <div style="display: flex; align-items: center;">
        <div>
            <h1>TD 5 - Likelihood and Parameter Estimation</h1>
            <h2>Understanding human behavior with cognitive models</h2>
            <h3>Master in Cognitive Science</h3>
            <h4>École Normale Supérieure - PSL</h4>
            <p> Valentin Wyart - Lecturer<br>
                Amric Trudel - Practical Sessions (TD)<br>
                <a href="mailto:amric.trudel@ens.psl.eu">amric.trudel@ens.psl.eu</a></p>
        </div>
        <div>
            <img src="images/logo_ens.png" style="height: 70px; margin-left: 10px;" />
        </div>
    </div>
</body>
</html>

# Objectives
The objective of this TD is to understand how to compute the likelihood of a behavioral trajectory given a computational cognitive model. Using the Reinforcement Learning model that we implemented last time, we will:
- Define the notion of likelihood for a single action (simplified)
- Define the likelihood of a full behavioral trajectory (simplified model)
- Write a log likelihood method for the RLModel class
- Explore likelihood values for different parameter values of an RLModel
- Implement Maximum Likelihood Estimation (MLE) to estimate the parameters of the model on behavioral data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pybads import BADS

from tests import test_rl_model_policy, test_likelihood_of_trajectory, test_log_likelihood_of_trajectory, test_rl_model_log_likelihood

%reload_ext autoreload
%autoreload 2

## Back to a simplified version of the Reinforcement Learning model
In the first part of this exercise, we will use a simplified version of the RLModel, which doesn't have a learning rule. This way, we can get an intution of the calculation of the likelihood without having to worry about q-value updates at first.

### Policy
Below is the `SimplifiedRLModel`, which has some of the methods of the `RLModel` class on which you worked last time. We will need to re-write the policy method. As a reminder, here is the formula for the softmax policy:

At each trial, the agent chooses the action that has the highest q-value, but with some randomness which depends on the degree of exploration that it is willing to do. The probability of choosing action 1 is given by the policy. This is mediated by the **temperature** parameter, where a model with a temperature of 0 will not explore at all and be 100% greedy.
$$
p_t = \frac{1}{1 + \exp(\frac{-(Q_t^1 - Q_t^0)}{\tau})}
$$
where:
- $p_t$ is the probability of choosing action 1 at trial $t$
- $Q_t^0$ and $Q_t^1$ are the q-values of actions 0 and 1 at trial $t$
- $\tau$ is the temperature

📝Implement the `policy` method in the `SimplifiedRLModel` class. Use the equation of the softmax rule given in the explantions.

In [ ]:
class SimplifiedRLModel:
    def __init__(self, learning_rate: float, temperature: float):
        self.learning_rate = learning_rate
        self.temperature = temperature
        self.q_values = np.array([0.5, 0.5]) # At t=0, both arms are assumed to have a value of 0.5

    def set_q_values(self, q_0: float, q_1: float):
        self.q_values = np.array([q_0, q_1])
        return self

    def policy(self):
        # Complete this code
        prob = ... # Use the softmax rule to compute the probability of choosing action 1
        return prob

Run this cell to test your implementation.

In [ ]:
test_rl_model_policy(SimplifiedRLModel)

### Likelihood of an action
Let's define the likelihood of an action given the model's parameters. The likelihood of an action is the probability that the model would have chosen this action given its parameters and the value of its latent variables. In bayesian terms, the likelihood of action $a_t$ at time $t$ is given by the following formula:
$$
\mathcal{L}(a_t) = p(a_t | \theta)
$$

📝Implement the `likelihood_of_action` function that takes a model and an action as input and returns the likelihood of the action given an instance of `SimplifiedRLModel`.

In [ ]:
def likelihood_of_action(model: SimplifiedRLModel, action: int) -> float:
    # your code here
    ...

📝Call your function with different parametrizations of the model and actions to see if it returns the expected values.
Try also to set different q-values by using the `set_q_values()` method.

In [ ]:
# Your code here
simple_model = ...
...

💭Among **parameters** and **latent variables** of the `SimplifiedRLModel`, list all the ones that influence the likelihood of an action.

_Your answer here_

### Likelihood of a full behavioral trajectory (Simplified model)
Now that we have an understanding of the likelihood of a single action, we can extend this to a full behavioral trajectory. Take some time to think about how you would compute the likelihood of a full behavioral trajectory given a model. What would be the formula to compute the likelihood of a full sequence of actions given a model?

💭Before you look at the code below, take a minutea to write down on a piece of paper the formula for the likelihood of a full behavioral trajectory given a model. Then write down the steps that you would take to implement this in code. It is worthwhile that you take the time to try to remember and write it down before you look at the answer. Ask for guidance if you are stuck or if you don't know where to start.

Yes, write it down! Really! 😁

### Likelihood of a trajectory

To make things simple, we will again assume that we are using our SimplifiedRLModel that doesn't process rewards nor update its q-values between trials. In this case, the likelihood of a full trajectory is simply defined as the **joint probability of all the actions of the trajectory** for the given model.

In other words, it is the product of the likelihood of each action. We will deal with the rewards and the update rule later.
Let's first implement the likelihood of a full trajectory given a simplified model, given the following formula:
$$
\mathcal{L}(a_1, a_2, ..., a_T) = \prod_{t=1}^{T} p(a_t | \theta)
$$

📝 Complete the `likelihood_of_trajectory` function below, that now takes as a second argument a sequence of actions (in a 1-dimensional array)

__Note__: You can re-use your `likelihood_of_action` function in this new function

In [ ]:
def likelihood_of_trajectory(model: SimplifiedRLModel, actions: np.ndarray) -> float:
    # Your code here
    ...

Test your function

In [ ]:
test_likelihood_of_trajectory(likelihood_of_trajectory)

📝 Run your function. Try to see what happens if you increase the number of trials.

In [ ]:
# Run your function
actions = np.random.choice([0, 1], size=10)
likelihood_of_trajectory(..., ...)

💭Do notice what happens when we compute the joint probability of an increasing number of actions? Do you remember what is the workaround found by mathematicians to avoid this?

_Your answer here_

### Log likelihood of a trajectory
Now, rewrite your function to compute the log likelihood of a full trajectory. This is a common practice because the likelihood of a full trajectory can be very small and difficult to work with. By taking the log, we can transform the product into a sum, which is easier to work with.
$$
\ell(a_1, a_2, ..., a_T) = \sum_{t=1}^{T} \log p(a_t | \theta)
$$

In [ ]:
def log_likelihood_of_trajectory(model: SimplifiedRLModel, actions: np.ndarray) -> float:
    # Your code here
    ...

Test your function

In [ ]:
test_log_likelihood_of_trajectory(log_likelihood_of_trajectory)

Run your function. Try to see what happens if you increase the number of trials.

In [ ]:
actions = np.random.choice([0, 1], size=10)
log_likelihood_of_trajectory(..., ...)

## Full RLModel (with learning rule)
Now that you understand the notion of log likelihood for a behavioral trajectory, we can get into the real stuff. An RLModel who plays a bandit task is normally able to process rewards and update the q-values of the actions at each trial. This means that the likelihood of a full trajectory is not only dependent on the actions taken, but also on the rewards obtained. 

To make things more handy, we will implement the log likelihood function as a method of the `RLModel` class. We will reuse the same class as the one you implemented during the last TD, along with its `update` method.

### Log likelihood of a full trajectory with rewards
To compute the log likelihood correctly, we therefore have to loop through the trials, update the q-values of the model, and compute the likelihood of each action given the updated q-values. 

$$
\ell(a_1, a_2, ..., a_T) = \prod_{t=1}^{T} p(a_t | \theta, r_{1:t})
$$
📝Write the `log_likelihood` method in the `RLModel` class. This method should take as input a sequence of actions and a sequence of rewards and return the log likelihood of the full trajectory.

In [ ]:
class RLModel:
    def __init__(self, learning_rate: float, temperature: float):
        self.learning_rate = learning_rate
        self.temperature = temperature
        self.q_values = np.array([0.5, 0.5])

    def policy(self):
        np.seterr(over='ignore') # We silence the warnings for numerical overflow
        prob = 1 / (1 + np.exp(-(self.q_values[1] - self.q_values[0]) / self.temperature))
        np.seterr(over='warn')
        return prob

    def update(self, action: int, reward: float):
        td_error = reward - self.q_values[action]
        self.q_values[action] += self.learning_rate * td_error

    def log_likelihood(self, actions: np.ndarray, rewards: np.ndarray) -> float:
        self.reset()
        # Your code here
        ...

    def simulate(self, task):
        # We don't need it today
        pass

    def reset(self):
        self.q_values = np.array([0.5, 0.5])

    def __repr__(self):
        return f'RLModel(lr={self.learning_rate: .3f}, t={self.temperature: .3f})'

Try your function manually to see if it seems to work

In [ ]:
n_trials = 40
sample_actions = np.random.choice([0, 1], size=n_trials)
sample_rewards = np.random.randn(n_trials)
model = RLModel(0.5, 0.5)

model.log_likelihood(..., ...)

Test your function with unit tests

In [ ]:
test_rl_model_log_likelihood(RLModel)

## Visualizing the likelihood on the parameter space
Let's see how the likelihood of a given behavior trajectory varies depending on the values of learning rate and temperature. For this we will produce a heatmap on the possible range of values for learning range and temperature.

First, let's load a sequence of actions and rewards that were were simulated with an RLModel with a pre-defined learning rate and temperature. You will try to guess what those values were.

In [ ]:
actions = np.load('actions.npy')
rewards = np.load('rewards.npy')

💭 How many trials were simulated in this data?

Then we define a sequence of `n_values` parameter values for each parameter that we will try.

📝 Define the bounds that will be explored for each parameter. Try to guess for now, and you will be able to change them later. Note that you can also explore a more restrictive range if you want.

In [ ]:
n_values = 40

lrs = np.linspace(..., ..., n_values)
temps = np.linspace(..., ..., n_values)

This code loops through all combinations of learning rates and temperatures.

**Attention:** We want to study the **likelihoods** here, so you need to convert the log likelihood returned by the `log_likelihood()` method of the RLModel class into a likelihood.

📝 Fill the missing parts indicated with comments

In [ ]:
likelihoods = pd.DataFrame()

for lr in lrs:
    for temp in temps:
        ... # Compute the likelihood of the behavior given the (lr, temp)
        likelihoods.loc[round(lr, 2), round(temp, 2)] = ... # Add your result to the dataframe

📝 Make a heatmap with the likelihood values made in your DataFrame. You an use the Seaborn library for that.

In [ ]:
# Your code here
...

💭 Can you guess the learning rate and temperature of the model that simulated this behavioral trajectory?

You can adjust the range of parameter values that you explore if you wish to better see the point of maximum likelihood.

## Maximum Likelihood Estimation (MLE)

There is a way to automate the parameter search that you just did manually!

The goal of Maximum Likelihood Estimation is to find the set of parameters that maximize the likelihood of the behavioral data given the model. In other words, that's what we do when we say that we "fit the model a subject's data". If we write this mathematically, we want to find the parameters $\theta$ that maximize the likelihood of the data $D$ given the model $M$:
$$
\begin{align}
\hat{\theta}_\text{MLE} &= \arg \max_\theta \ell(\text{behavior} | \theta, s) \\
                        &= \arg \max_{\theta}(\log p(\text{behavior} | \theta, s))      
\end{align}
$$


### Optimization routine
Now let's write the code that will find the set of parameters that maximize the likelihood of the behavioral data. For this, we can use a standard optimizer that will do the job for us. We will use BADS (Bayes Adaptive Direct Search). All we need to do is provide it with details about the range of values that we want to explore for each parameter, and a cost function that it needs to minimize. An optimization problem can either be framed as the _maximization_ or _minimization_ of an objective function, but it's standard for off-the-shelf optimizers to **minimize**. Since we have just defined a `model_performance` that we wish to **maximize**, we will have to add a minus sign to turn it into a `cost function` that we wish to **minimize**.

📝 Complete the `optimize_rl_model()` function:
1. Write the `cost_function`, that gives the "cost" associated with an RLModel with the given parameters:
    -  Instantiate an RLModel with the parameters given as arguments
    -  Compute the log likelihood of the actions passed to `fit_rl_model()` given this model and the `rewards` passed to `fit_rl_model()`
    -  Return the cost as a value that we wish to **minimize**

2. Complete the settings of the BADS optimizer, each of which is a list with two values, one for each parameter in the following order: \[lr, temp]. I let you think about what settings could be appropriate and we'll discuss it together. As a reference:
    - `x0` represent the typical value that you would expect for each parameter. It's the starting point of the optimization.
    - `lower_bounds` and `upper_bounds` specify the range of values that will be explored. In our case we have bounds of $[0, 1]$ for both parameters
    - `plausible_lower_bound` and `plausible_upper_bound` are more restrictive than the actual bounds and specify the range where we expect to find most solutions.

In [ ]:

def fit_rl_model(actions: np.ndarray[int], rewards: np.ndarray[float],
                     verbose: bool = True):
    
    ##########################################################################
    # In python we can define functions inside functions.
    # Here we define the cost function that the BADS optimizer will try to minimize
    # It takes as input the parameters of the model and returns a value that we wish to minimize
    def cost_function(params):
        learning_rate, temperature = params
        # Your code here
        ...
        ...
        return ...
    ############################################################################

    optimizer = BADS(
        fun=cost_function,
        x0=[..., ...], # Fill this [lr, temp],
        lower_bounds=[..., ...], # Fill this
        upper_bounds=[..., ...], # Fill this
        plausible_lower_bounds=[..., ...], # Fill this
        plausible_upper_bounds=[..., ...], # Fill this
        options={
            'display': 'iter' if verbose else 'off'
        }
    )
    result = optimizer.optimize()
    optimal_params = {
        'learning_rate': result.x[0],
        'temperature': result.x[1]
    }
    return RLModel(**optimal_params)

📝 Run the optimization on the same actions and rewards as in the last exercise and see what value you find out!

In [ ]:
actions = np.load('actions.npy')
rewards = np.load('rewards.npy')

... # Your code here

## 💪 Optional exercise: Impact of the number of trials
Try to see what happens if you shorten the length of the sequence of actions and rewards. What parameters do you find? To what extent do you get values that are far away from the ones you get with the full trajectory?

In [ ]:
n_trials = ...

partial_actions = ...
partial_rewards = ...

fitted_model_partial = ...